# Notebook 07a: Venue-Year Source Resolution — Tiered Fallback Strategy

Clean rebuild, replacing the 06 / 06a / 06b chain (kept for reference, but this is now the canonical version).

**Strategy (per supervisor guidance):**
1. **Tier 1 — Structured source lookup:** use `primary_location.source.id` from the paper's OpenAlex work record (any type accepted, not just `"conference"` — this was the 06b fix).
2. **Tier 2 — Raw name fallback:** when Tier 1 returns null, read the `raw_source_name` field (present even when structured `source` fails to resolve) and search `/sources?search=<raw_source_name>` for a candidate match.
3. **Validation guardrail:** reject Tier 2 candidates whose `type` is `repository`/`institution`, or whose name doesn't share enough tokens with the expected conference name — this prevents mismatches like "AAAI '14" resolving to "University of Alberta".
4. **Remainder:** anything still unresolved after both tiers is flagged for manual review, not silently dropped.


> **Renamed to 07a** to guarantee a fresh, fully-complete file — avoids any risk of reopening a stale/partial download of `07`.

In [ ]:
import pandas as pd
import os, json, time, re, requests
from collections import Counter

MAILTO = 'research@example.com'  # replace with actual contact email for polite pool
os.makedirs('/home/user/output', exist_ok=True)

df = pd.read_csv('huang_matched_openalex.csv')
df_with_id = df[df['openalex_id'].notna()].copy()
df_with_id['openalex_id'] = df_with_id['openalex_id'].astype(str)

def extract_work_id(url_or_id):
    return url_or_id.rstrip('/').split('/')[-1]

grouped = df_with_id.groupby(['conference','year'])['openalex_id'].apply(list).to_dict()
print(len(grouped), "total (conference, year) pairs")


## Seed Tier-1 Results From 06b

Reuse the 164 (or however many) pairs 06b already resolved successfully — no need to re-fetch those.

In [ ]:
CACHE_FILE = '/home/user/output/nb07a_venue_year_sources_cache.json'
ERROR_LOG_FILE = '/home/user/output/nb07a_error_log.json'

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        venue_year_cache = json.load(f)
    print(f"CACHE HIT: {len(venue_year_cache)} pairs already in nb07 cache")
else:
    venue_year_cache = {}
    OLD_CACHE_FILE = '/home/user/output/nb06b_venue_year_sources_cache.json'
    if os.path.exists(OLD_CACHE_FILE):
        with open(OLD_CACHE_FILE) as f:
            old_cache = json.load(f)
        seeded = 0
        for key, val in old_cache.items():
            if val.get('n_votes', 0) > 0:
                val['tier'] = 1
                venue_year_cache[key] = val
                seeded += 1
        print(f"Seeded {seeded} Tier-1 successful pairs from 06b")

if os.path.exists(ERROR_LOG_FILE):
    with open(ERROR_LOG_FILE) as f:
        error_log = json.load(f)
else:
    error_log = {}

remaining_keys = [f"{conf}|{year}" for (conf, year) in grouped if f"{conf}|{year}" not in venue_year_cache]
print(len(remaining_keys), "pairs still need resolution (Tier 1 fresh attempt + Tier 2 fallback)")


## Tier 1: Structured Source Lookup

Same relaxed logic as 06b — accept any source type, not just `conference`. Also captures `raw_source_name` from the work record so Tier 2 has it on hand without an extra API call.

In [ ]:
def fetch_work_record(wid, key, sample_idx):
    """Fetch a single work. Returns dict with source info + raw_source_name, or None on failure."""
    try:
        r = requests.get(
            f"https://api.openalex.org/works/{wid}",
            params={'mailto': MAILTO},
            timeout=25
        )
        if r.status_code == 429:
            error_log.setdefault(key, []).append({'sample': sample_idx, 'error': 'RATE_LIMITED_429'})
            time.sleep(2)
            return None
        r.raise_for_status()
        w = r.json()

        primary = w.get('primary_location', {}) or {}
        src = primary.get('source')
        raw_name = primary.get('raw_source_name')

        if src and src.get('id'):
            return {'id': src['id'], 'name': src['display_name'], 'source_type': src.get('type'),
                     'raw_source_name': raw_name, 'tier_used': 1}

        for loc in w.get('locations', []):
            s = loc.get('source')
            if s and s.get('id'):
                return {'id': s['id'], 'name': s['display_name'], 'source_type': s.get('type'),
                         'raw_source_name': raw_name or loc.get('raw_source_name'), 'tier_used': 1}
            if not raw_name and loc.get('raw_source_name'):
                raw_name = loc.get('raw_source_name')

        # Tier 1 failed structurally, but we may still have a raw_source_name to fall back on
        error_log.setdefault(key, []).append({'sample': sample_idx, 'error': 'NO_STRUCTURED_SOURCE'})
        return {'id': None, 'name': None, 'source_type': None, 'raw_source_name': raw_name, 'tier_used': None}

    except requests.exceptions.Timeout:
        error_log.setdefault(key, []).append({'sample': sample_idx, 'error': 'TIMEOUT'})
        return None
    except requests.exceptions.RequestException as e:
        error_log.setdefault(key, []).append({'sample': sample_idx, 'error': f'REQUEST_ERROR: {repr(e)}'})
        return None
    except Exception as e:
        error_log.setdefault(key, []).append({'sample': sample_idx, 'error': f'UNKNOWN_ERROR: {repr(e)}'})
        return None


## Tier 2: Raw Name Fallback + Validation Guardrail

When Tier 1 fails, search `/sources?search=<raw_source_name>` for a candidate. Before accepting it:
- Reject candidates typed `repository` or that have `type` indicating an institution/non-venue entity.
- Require meaningful token overlap between the candidate's `display_name` (and `alternate_titles`) and the expected conference name — this is the guardrail against cases like "AAAI '14" → "University of Alberta".

In [ ]:
STOPWORDS = {'the','of','on','and','for','in','international','conference','workshop','symposium','annual','proceedings'}

def tokenize(name):
    tokens = re.findall(r"[a-zA-Z]+", name.lower())
    return {t for t in tokens if t not in STOPWORDS and len(t) > 1}

def token_overlap_score(candidate_name, conf_shorthand, candidate_alt_titles=None):
    cand_tokens = tokenize(candidate_name)
    if candidate_alt_titles:
        for alt in candidate_alt_titles:
            cand_tokens |= tokenize(alt)
    conf_tokens = tokenize(conf_shorthand)
    if not conf_tokens:
        return 0.0
    # also check if shorthand acronym appears literally in candidate name (case-insensitive)
    acronym_hit = conf_shorthand.lower() in candidate_name.lower()
    overlap = len(cand_tokens & conf_tokens) / max(len(conf_tokens), 1)
    return max(overlap, 0.6 if acronym_hit else 0.0)

REJECT_TYPES = {'repository', 'institution', 'funder', 'publisher'}

def search_source_by_name(raw_name, conf_shorthand, key, min_score=0.5):
    """Tier 2 fallback: search OpenAlex sources by raw_source_name, validate against expected conference."""
    if not raw_name:
        error_log.setdefault(key, []).append({'error': 'TIER2_NO_RAW_NAME'})
        return None
    try:
        r = requests.get(
            "https://api.openalex.org/sources",
            params={'search': raw_name, 'per_page': 5, 'mailto': MAILTO},
            timeout=25
        )
        r.raise_for_status()
        results = r.json().get('results', [])
        if not results:
            error_log.setdefault(key, []).append({'error': 'TIER2_NO_SEARCH_RESULTS', 'raw_name': raw_name})
            return None

        best = None
        best_score = 0.0
        for cand in results:
            ctype = cand.get('type')
            if ctype in REJECT_TYPES:
                continue
            score = token_overlap_score(cand.get('display_name',''), conf_shorthand,
                                          cand.get('alternate_titles'))
            if score > best_score:
                best_score = score
                best = cand

        if best and best_score >= min_score:
            return {'id': best['id'], 'name': best['display_name'], 'source_type': best.get('type'),
                     'raw_source_name': raw_name, 'tier_used': 2, 'match_score': round(best_score, 2)}
        else:
            error_log.setdefault(key, []).append({'error': 'TIER2_VALIDATION_FAILED',
                                                     'raw_name': raw_name, 'best_score': round(best_score,2)})
            return None
    except requests.exceptions.RequestException as e:
        error_log.setdefault(key, []).append({'error': f'TIER2_REQUEST_ERROR: {repr(e)}'})
        return None


## Resolve Pair: Tier 1 With Tier 2 Fallback, Voting Across Samples

In [ ]:
def resolve_pair(conf, year, urls, max_samples=3):
    key = f"{conf}|{year}"
    candidates = urls[:max_samples]
    votes = []
    raw_names_seen = []

    for i, url in enumerate(candidates):
        wid = extract_work_id(url)
        result = fetch_work_record(wid, key, i)
        time.sleep(0.15)
        if result is None:
            continue
        if result.get('id'):
            votes.append(result)
        elif result.get('raw_source_name'):
            raw_names_seen.append(result['raw_source_name'])

    if votes:
        counts = Counter(v['id'] for v in votes)
        top_id, _ = counts.most_common(1)[0]
        top = next(v for v in votes if v['id'] == top_id)
        return {'source_id': top_id, 'source_name': top['name'], 'source_type': top.get('source_type'),
                 'tier_used': 1, 'n_votes': len(votes), 'match_score': None}

    # Tier 1 fully failed for all samples — try Tier 2 using any raw_source_name we captured
    for raw_name in raw_names_seen:
        result = search_source_by_name(raw_name, conf, key)
        time.sleep(0.15)
        if result:
            return {'source_id': result['id'], 'source_name': result['name'], 'source_type': result.get('source_type'),
                     'tier_used': 2, 'n_votes': 1, 'match_score': result.get('match_score')}

    return {'source_id': None, 'source_name': None, 'source_type': None,
             'tier_used': None, 'n_votes': 0, 'match_score': None}


## Run Resolution Over Remaining Pairs

Checkpoints to disk after every pair so progress is never lost.

In [ ]:
processed = 0
for key in remaining_keys:
    conf, year = key.split('|')
    year = int(year)
    urls = grouped.get((conf, year), [])
    result = resolve_pair(conf, year, urls)
    venue_year_cache[key] = result
    with open(CACHE_FILE, 'w') as f:
        json.dump(venue_year_cache, f, indent=2)
    with open(ERROR_LOG_FILE, 'w') as f:
        json.dump(error_log, f, indent=2)
    processed += 1
    if processed % 10 == 0:
        print(f"Progress: {processed}/{len(remaining_keys)} newly processed | Total cached: {len(venue_year_cache)}/{len(grouped)}")

print(f"\nDone. Processed {processed} new pairs this run.")


## Tier Breakdown and Remaining Nulls

In [ ]:
tier_counts = Counter(v.get('tier_used') for v in venue_year_cache.values())
still_null = [k for k, v in venue_year_cache.items() if v.get('source_id') is None]

print("Tier 1 resolved:", tier_counts.get(1, 0))
print("Tier 2 resolved:", tier_counts.get(2, 0))
print("Still unresolved (flagged for manual review):", len(still_null))

if still_null:
    print("\nSample of unresolved pairs:", still_null[:10])


## Final Lookup Table

In [ ]:
rows = []
for key, val in venue_year_cache.items():
    conf, year = key.split('|')
    rows.append({
        'conference': conf,
        'year': int(year),
        'source_id': val.get('source_id'),
        'source_name': val.get('source_name'),
        'source_type': val.get('source_type'),
        'tier_used': val.get('tier_used'),
        'n_votes': val.get('n_votes'),
        'match_score': val.get('match_score'),
        'needs_manual_review': val.get('source_id') is None
    })

lookup_df = pd.DataFrame(rows).sort_values(['conference','year']).reset_index(drop=True)
lookup_df.to_csv('/home/user/output/nb07a_venue_year_source_lookup.csv', index=False)
print(lookup_df.shape)
lookup_df.head(10)


> **Note on manual-review rows:** rows with `needs_manual_review == True` failed both Tier 1 (no structured source on any sampled paper) and Tier 2 (either no raw_source_name available, or the name-search match score fell below the validation threshold). These should be spot-checked by hand rather than silently dropped, per supervisor guidance — the validation guardrail is intentionally conservative to avoid mismatches like AAAI '14 resolving to an unrelated institution.

## Next Step (Notebook 08)
Use `nb07_venue_year_source_lookup.csv` as the definitive venue-year-to-OpenAlex-source mapping. Manually resolve the small remainder flagged `needs_manual_review`, then proceed to citation/lift analysis.
